# Importing a public dataset : free exploration of naive larvae

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

An entry point to **larvaworld** built around a published dataset of freely crawling
*Drosophila* larvae. The notebook downloads the data itself, imports it, analyses it and
visualizes it; running the cells is the only thing you have to do.

**The dataset.** Locomotion of naive third-instar larvae recorded by the Schleyer lab and
published on G-Node, tracked as a twelve-point midline with the body contour. It contains
**31 recording dishes**, each holding one file per tracked animal. There is no stimulus and
no experimental manipulation - the larvae simply explore an empty dish.

**The question.** With a single, unmanipulated condition there is no group contrast to
test. What such a dataset is for is a **baseline**: how a naive larva crawls when nothing is
done to it. We therefore import it twice, as the Schleyer example in `import_datasets.ipynb`
does - once from a **single dish** and once **pooled across all dishes** - and compare the
two, which asks a question worth asking of any tracking dataset: *is one dish
representative of the whole experiment?*

This notebook is one of a series; a blank version is available as
`import_public_dataset_template.ipynb`.

## Setup

Importing larvaworld initializes its configuration registry : some components are loaded
from disc and the rest are built on the fly. `VERBOSE = 1` makes the import report what it
is doing, which is worth watching the first time.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython.display import display

import larvaworld
from larvaworld.lib import reg, sim, util
from larvaworld.lib.reg.generators import ReplayConf

larvaworld.VERBOSE = 1

# The name of this experiment. It labels the imported datasets and the output folders.
EXPERIMENT_NAME = "FreeExploration"

MEDIA_DIR = Path(f"./media/{EXPERIMENT_NAME}")
plot_dir = (MEDIA_DIR / "plots").as_posix()
video_dir = (MEDIA_DIR / "videos").as_posix()

# Rendering the replay videos needs ffmpeg and takes several minutes.
MAKE_VIDEOS = False

ds = []  # the imported datasets, filled in further below

import shutil
import zipfile

import requests


def fetch_dataset(expect, archive, url=None, extract=True):
    """Make a dataset available locally, doing as little work as possible.

    Resolves in three steps, reporting which one it took :
      1. the extracted data is already there  -> nothing happens
      2. the archive is there but not unpacked -> unpack only
      3. neither                               -> download, then unpack

    Args:
        expect: path that exists once the data is unpacked.
        archive: path of the downloaded archive.
        url: where to download the archive from, if it is missing.
        extract: whether the archive can be unpacked here. RAR archives cannot.

    Returns:
        True if `expect` is available afterwards.
    """
    expect, archive = Path(expect), Path(archive)
    if expect.exists():
        print(f"[1/3] already present, nothing to do : {expect}")
        return True

    if not archive.exists():
        if url is None:
            print(f"[3/3] missing and no download link given : {archive}")
            return False
        archive.parent.mkdir(parents=True, exist_ok=True)
        print(f"[3/3] downloading {url}\n      -> {archive}")
        tmp = archive.with_suffix(archive.suffix + ".part")
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            done = 0
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    done += len(chunk)
                    if total:
                        print(
                            f"      {done / 1e6:8.0f} / {total / 1e6:.0f} MB", end="\r"
                        )
        tmp.rename(archive)
        print(f"\n      downloaded {archive.stat().st_size / 1e6:.0f} MB")
    else:
        print(f"[2/3] archive already downloaded : {archive}")

    if not extract:
        print(f"      this archive cannot be unpacked from Python. Extract it with")
        print(f"      7-Zip, WinRAR or unrar so that this exists :\n      {expect}")
        return expect.exists()

    print(f"      unpacking -> {expect.parent}")
    with zipfile.ZipFile(archive) as z:
        z.extractall(expect.parent)
    return expect.exists()

## Section 1 : Get the data

The dataset is openly available on G-Node.

- **Record** : <https://doi.org/10.12751/g-node.5e1ifd>
- **Archive** : <https://doi.gin.g-node.org/10.12751/g-node.5e1ifd/10.12751_g-node.5e1ifd.zip>

The cell below fetches it. It is a **935 MB** download, and the archive contains a second
archive inside it, so it is unpacked twice. Nothing is downloaded or unpacked if it is
already on disc, so re-running the notebook is free.

In [ ]:
DOWNLOAD_ROOT = Path.home() / "Downloads" / "10.12751_g-node.5e1ifd"
URL = "https://doi.gin.g-node.org/10.12751/g-node.5e1ifd/10.12751_g-node.5e1ifd.zip"

# The outer archive unpacks into a folder holding a second archive ...
fetch_dataset(
    expect=DOWNLOAD_ROOT,
    archive=DOWNLOAD_ROOT.with_suffix(".zip"),
    url=URL,
)
# ... which holds the recordings themselves.
INNER = DOWNLOAD_ROOT / "Naive_locomotion_Drosophila_larvae"
fetch_dataset(expect=INNER, archive=INNER.with_suffix(".zip"))

RAW_FOLDER = DOWNLOAD_ROOT.as_posix()
EXPERIMENT = "Naive_locomotion_Drosophila_larvae"

DATA_AVAILABLE = INNER.is_dir()
if DATA_AVAILABLE:
    dishes = sorted(p.name for p in INNER.iterdir() if p.is_dir())
    print(f"\n{len(dishes)} recording dishes, e.g. {dishes[0]}")
    print(f"{len(list(INNER.rglob('*.csv')))} tracked animals in total")

## Section 2 : Import to larvaworld

Three things have to be specified : **where the data is**, **which tracker wrote it**, and
**which tracks to keep**.

### What the tracker recorded, and what it did not

Before importing, it is worth knowing which properties of a recording are written down
somewhere and which are not. For most published tracking data the picture is this :

| property | stated in the archive? | larvaworld can derive it |
|---|---|---|
| recording duration | usually, in the tracker's settings | not needed |
| stimulus protocol | usually, in the tracker's settings | not needed |
| **frame rate** | **often not** | **yes**, from the timestamps |
| **number of midline points** | **no** | **yes**, from the coordinates |
| **arena dimensions** | **no** | partly, see below |
| **pixel-to-millimetre scale** | **no** | no - you have to know it |

The highlighted rows are the ones that matter for the import, and they are the ones least
likely to be recorded. larvaworld therefore derives what it can from the data itself :

- **Frame rate.** Many trackers record at a variable rate, so a single nominal frame rate
  does not describe them. When a lab format declares a variable framerate, the timestep is
  measured from the timestamps and used for the whole import, including the stored dataset.
  Pass `estimate_dt=False` to keep the nominal value. Formats whose data carries no
  timestamps at all cannot use this, and need the frame rate set by hand.
- **Midline points.** Counted from the data and used whenever it disagrees with the
  expected number. Pass `estimate_midline_points=False` to switch this off.
- **Arena dimensions.** Estimated from the area the animals actually covered, which makes
  it a *lower bound* : larvae that never reach the rim make the arena look smaller than it
  is. It is therefore **off by default**. Pass `estimate_arena_dimensions=True`.

**The spatial scale is the one thing you must bring yourself.** If a tracker exports pixels
rather than millimetres, nothing in the coordinates reveals the conversion factor. A quick
sanity check settles which case you are in : the summed length of a larva's midline should
be a few millimetres for a third-instar larva. larvaworld applies that same check on
import and refuses data implying an impossible animal.

### Which tracker wrote it

This data comes from the Schleyer lab. That format records at a constant frame rate and
tracks the contour as well as the midline, so nothing about its timing has to be measured
from the data.

In [ ]:
lf = reg.conf.LabFormat.get("Schleyer")

### Which tracks to keep, and what to compute

Only tracks longer than a couple of minutes are kept, since a fragment says nothing about
exploration. The annotation set is the full one, because the stride and turn plots further
down are built from it - `interference` in particular is what produces the per-stride
metrics.

In [ ]:
constraints = util.AttrDict({"min_duration_in_sec": 120})

enr_kws = util.AttrDict(
    {
        "proc_keys": ["angular", "spatial"],
        "anot_keys": ["bout_detection", "bout_distribution", "interference"],
        "traj2origin": True,
        "tor_durs": [20],
        "dsp_starts": [0],
        "dsp_stops": [40, 60],
    }
)

### The two datasets

One dish on its own, and all dishes pooled. The pooled import is capped so that the two are
of comparable size and the figures stay readable.

In [ ]:
DISH = "box1-2017-05-18_14_48_22"  # any of the dishes listed above
N_POOLED = 50

common = {
    "raw_folder": RAW_FOLDER,
    "group_id": EXPERIMENT_NAME,
    "save_dataset": True,
    "enrich_conf": enr_kws,
    **constraints,
}
refIDs = [f"{EXPERIMENT_NAME}.single_dish", f"{EXPERIMENT_NAME}.pooled"]

if DATA_AVAILABLE:
    ds = [
        lf.import_dataset(
            parent_dir=f"{EXPERIMENT}/{DISH}",
            id="single_dish",
            refID=refIDs[0],
            color="black",
            **common,
        ),
        lf.import_dataset(
            parent_dir=EXPERIMENT,
            merged=True,
            max_Nagents=N_POOLED,
            id="pooled",
            refID=refIDs[1],
            color="red",
            **common,
        ),
    ]
    for d in ds:
        print(
            f"{d.id:12s} : {d.config.N} larvae, dt={d.config.dt:.4f} s, "
            f"{d.config.Npoints} midline points"
        )

### Reloading in a later session

From now on you never touch the downloaded archive again.

In [ ]:
if not ds:
    if all(refID in reg.conf.Ref.confIDs for refID in refIDs):
        ds = [reg.loadRef(id=refID, load=True) for refID in refIDs]
        print("Loaded :", [d.id for d in ds])
    else:
        print("These datasets have not been imported yet. Run Section 2 first.")

## Section 3 : Data analysis and plotting

larvaworld ships a library of plotting routines, each registered under a short name. You
pick one by name and hand it the datasets you want compared - the group colors and labels
are taken from the datasets themselves, so every figure is consistent.

In [ ]:
# The available plots, by their unique IDs
print(reg.graphs.ks)

In [ ]:
# Arguments shared by every plot below. Figures are also written to `plot_dir`.
plot_kws = {"datasets": ds, "save_to": plot_dir, "show": False, "subfolder": None}

### The trajectories

First, simply what the larvae did : their paths over the analysed window.

In [ ]:
if ds:
    display(reg.graphs.run("trajectories", **plot_kws))

The same trajectories, but each one translated so that it starts at the origin, and colored
by group. This removes the arbitrary starting position of each animal and makes the *shape
and extent* of the paths directly comparable.

In [ ]:
if ds:
    display(
        reg.graphs.run("trajectories", mode="origin", single_color=True, **plot_kws)
    )

### Endpoint metrics

A boxplot of endpoint metrics - one value per larva, summarising its whole track. Each plot
routine has a default selection, but you can always name the metrics you want by their
short keys, as done here.

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=[
                "l",
                "fsv",
                "sv_mu",
                "str_sd_mu",
                "run_tr",
                "pau_tr",
                "tor20_mu",
                "dsp_0_40_fin",
                "b_mu",
                "bv_mu",
            ],
            **plot_kws,
        )
    )

And a composite figure summarising exploration behavior across the groups.

In [ ]:
if ds:
    display(reg.graphs.run("exploration summary", **plot_kws))

### Crawling and turning

Two overview figures of the parameters that describe the two things a crawling larva does : the forward strides and the lateral bending that steers them.

In [ ]:
if ds:
    display(reg.graphs.run("crawl pars", **plot_kws))

In [ ]:
if ds:
    display(reg.graphs.run("angular pars", **plot_kws))

### Individual strides and turns

Zooming in from distributions to single events. These are built from the bout annotation requested above : each figure marks the detected strides or turns on a stretch of one larva's track, which is the quickest way to see whether the detection is behaving sensibly on your data.

In [ ]:
if ds:
    display(reg.graphs.run("stride track", **plot_kws))

In [ ]:
if ds:
    display(reg.graphs.run("turn track", **plot_kws))

### Dispersal

Dispersal is the distance of a larva from where it started. We compare the single dish and the pooled set on it
in three increasingly informative ways.

**1. As an endpoint statistic.** The mean, final and maximum dispersal reached during the
analysed window - one number per larva, summarised as a boxplot per group.

In [ ]:
if ds:
    display(
        reg.graphs.run(
            "endpoint box",
            ks=["dsp_0_60_mu", "dsp_0_60_fin", "dsp_0_60_max"],
            **plot_kws,
        )
    )

**2. As a timecourse.** Dispersal plotted against time, showing both the mean and the
variance of each group : not just how far the groups got, but how fast and how consistently.

In [ ]:
if ds:
    display(reg.graphs.run("dispersal", **plot_kws))

In [ ]:
if ds:
    display(reg.graphs.run("dispersal", range=(0, 60), **plot_kws))

**3. Alongside the paths that produced it.** The summary versions place the timecourse next
to the corresponding trajectories, which makes the link between curve and behavior
immediate.

In [ ]:
if ds:
    display(reg.graphs.run("dispersal summary", **plot_kws))

## Section 4 : Visualize the dataset

A *replay* is a simulation whose agents are driven by recorded data instead of a model. It
gives you the same visualization tools you would use on a simulation - here, the
trajectories of all larvae of a group, transposed to a common origin and drawn as
accumulating trails.

Rendering needs `ffmpeg` (installed with larvaworld via `imageio_ffmpeg`) and takes a few
minutes per group, so it is off by default. Set `MAKE_VIDEOS = True` in the Setup cell.

In [ ]:
def run_replay(d):
    """Render one dataset's tracks to a video file in `video_dir`."""
    screen_kws = {
        "vis_mode": "video",
        "show_display": False,
        "draw_contour": False,
        "draw_midline": False,
        "draw_centroid": False,
        "visible_trails": True,
        "save_video": True,
        "fps": 1,
        "video_file": d.id,
        "media_dir": video_dir,
    }
    replay_conf = ReplayConf(
        transposition="origin", time_range=(0, 60), track_point=d.c.point_idx
    ).nestedConf
    rep = sim.ReplayRun(
        dataset=d,
        parameters=replay_conf,
        id=f"{d.id}_replay",
        screen_kws=screen_kws,
    )
    return rep.run()

In [ ]:
if MAKE_VIDEOS and ds:
    for d in ds:
        run_replay(d)

Finally the videos are stacked side by side into a single one, giving a direct visual
comparison of the groups.

In [ ]:
if MAKE_VIDEOS and ds:
    from larvaworld.lib.util.combining import combine_videos

    combine_videos(file_dir=video_dir, save_as="combined.mp4")
    print(f"Written to {video_dir}/combined.mp4")

## A few words on the lab format

Every tracker writes its own files, so larvaworld reads each one through a named **lab
format**. A lab format knows how a lab's files are laid out and how their contents must be
preprocessed, which is why the import above needed nothing more than a folder and a name.

| lab format | suits data that looks like |
|---|---|
| `Jovanic` | one file per recorded quantity, all animals stacked together |
| `Schleyer` | one file per animal, plus per-dish metadata |
| `Berni`, `Arguello` | one file per animal, columns in a fixed order |
| `DeepLabCut` | DeepLabCut CSV/HDF5 exports, one file per video |

Two things follow from this :

- **If one of them matches your tracker**, this notebook works on your own data with only
  the first section changed.
- **If none does**, a new lab format can be described and registered, after which your data
  imports like any other.

A lab format carries nominal values for things like the frame rate and the arena, because
they are usually constant for a lab. They describe the lab, not any particular recording,
which is why the import prefers what it can measure in the data itself.

## References

> Thoener J, Schleyer M (2021) *Locomotion of naive Drosophila larvae*. G-Node.
> <https://doi.org/10.12751/g-node.5e1ifd>

Please cite the dataset if you use it.